# Demo

I am recreating the important parts of the [dm_control_suite](https://github.com/google-deepmind/mujoco_playground/blob/main/learning/notebooks/dm_control_suite.ipynb) example.

In [1]:



import sys
print(sys.executable)

import jax
print("jax:", jax.__version__)

/bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/bin/python
jax: 0.8.2


In [5]:
%env MUJOCO_GL=egl

import os
import mujoco
import numpy as np

import jax
import mediapy as media
from tqdm import tqdm
import twmr  # noqa: F401
import warp as wp
from jax import numpy as jp
from mujoco_playground import registry

wp.config.quiet = True

media.set_ffmpeg("/bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.pixi/envs/default/bin/ffmpeg")

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get("XLA_FLAGS", "")
xla_flags += " --xla_gpu_triton_gemm_any=True"
os.environ["XLA_FLAGS"] = xla_flags

# env = registry.load("TransformableWheelMobileRobot")
env = registry.load("TWMRLegFlat")

state = env.reset(jax.random.PRNGKey(0))

# action = jp.ones(env.action_size) * 0.01
# action = jp.ones(env.action_size) * 2
action = jp.array([0.0, 0.0, 0.0, 0.0, 3.0, 3.0, 3.0, 3.0])

# action = jp.zeros(env.action_size)
# TODO: state.data.ctrl... what is the difference?

rollout = [state]
for _ in tqdm(range(500), desc="Simulating steps"):
    state = env.step(state=state, action=action)
    rollout.append(state)

print(f"Final position: {state.data.qpos}")

env: MUJOCO_GL=egl


Simulating steps: 100%|██████████| 500/500 [05:02<00:00,  1.65it/s]

Final position: [ 5.3982258e-02  3.4299698e-11  3.4865070e-02  1.0000000e+00
  1.0438179e-10  4.6342901e-08  3.8047521e-09 -4.5730665e-02
  3.4340122e+00  3.4337504e+00  3.4337463e+00 -4.5730758e-02
  3.4340122e+00  3.4337504e+00  3.4337463e+00 -2.5880683e-02
  3.4368935e+00  3.4366779e+00  3.4366732e+00 -2.5880642e-02
  3.4368935e+00  3.4366782e+00  3.4366732e+00]


In [6]:
# ── Adjustable camera parameter ──────────────────────────────────────────────
camera_distance = 1.5  # meters; increase to zoom out, decrease to zoom in
# ─────────────────────────────────────────────────────────────────────────────


def render_with_camera(env, rollout, camera_distance=1.5, height=480, width=640):
    """Render a rollout with a follow-camera at a configurable distance.

    The camera tracks the chassis body, staying `camera_distance` metres away,
    providing a stable third-person view that is closer than the default.
    """
    mj_model = env.mj_model
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(mj_model, cam)
    cam.azimuth = 135.0    # 3/4 rear view
    cam.elevation = -20.0  # slight downward angle
    cam.distance = camera_distance

    chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
    if chassis_id < 0:
        chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "root")
    if chassis_id < 0:
        chassis_id = 1  # fallback to first non-world body

    frames = []
    d = mujoco.MjData(mj_model)
    for state in tqdm(rollout, desc="Rendering frames"):
        d.qpos[:] = np.array(state.data.qpos)
        d.qvel[:] = np.array(state.data.qvel)
        d.mocap_pos[:] = np.array(state.data.mocap_pos)
        d.mocap_quat[:] = np.array(state.data.mocap_quat)
        d.xfrc_applied[:] = np.array(state.data.xfrc_applied)
        mujoco.mj_forward(mj_model, d)

        cam.lookat[:] = d.xpos[chassis_id]
        renderer.update_scene(d, camera=cam)
        frames.append(renderer.render())

    renderer.close()
    return frames


frames = render_with_camera(env, rollout, camera_distance=camera_distance)

media.show_video(frames, fps=1.0 / env.dt)
media.write_video("tlwr_demo.mp4", frames, fps=1.0 / env.dt)

Rendering frames: 100%|██████████| 501/501 [00:00<00:00, 536.04it/s]


This browser does not support the video tag.


In [4]:
# env_cfg = registry.get_default_config("TransformableWheelMobileRobot")
env_cfg = registry.get_default_config("TWMRLegFlat")
